In [0]:
# Load raw data with latin-1 encoding to handle special characters
# Convert Spark DataFrame to Pandas for EDA work
# .toPandas() collects all data from the cluster to the driver node
df = spark.read.table("novasend_fulfillment.raw.data_co_supply_chain_dataset").toPandas()

# Confirm shape and that data loaded correctly
print(df.shape)
df.head()
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

In [0]:
# Target Variable Count
print(df["Late_delivery_risk"].value_counts())
print()
print(df["Late_delivery_risk"].value_counts(normalize=True).round(3))

In [0]:
# Shows column names, non-null counts, and dtypes
df.info()

In [0]:
# Full list of columns to drop before any analysis or modeling
cols_to_drop = [
    # Leakage: recorded post-delivery
    "Days for shipping (real)",
    "Delivery Status",
    "Benefit per order",
    "Order Profit Per Order",

    # Peronsal Identifiers: no predictive value
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Password",

    # ID columns: no predictive signal
    "Customer Id",
    "Order Customer Id",
    "Order Id",
    "Order Item Id",
    "Order Item Cardprod Id",
    "Product Card Id",

    # Redundant IDs where name column already exists
    "Category Id",
    "Department Id",
    "Product Category Id",

    # Empty or non-tabular
    "Product Description",
    "Product Image",

    # Store location: Order Region and Market cover destination geography better
    "Latitude",
    "Longitude",

    # Too sparse or granular to model reliably
    "Customer Street",
    "Customer Zipcode",
    "Order Zipcode",
]

# Drop all at once and confirm what's left
df_clean = df.drop(columns=cols_to_drop)

print(f"Columns remaining: {df_clean.shape[1]}")
print(df_clean.columns.tolist())

In [0]:
# Drop profit ratio and shipping date: not available at order intake
df_clean = df_clean.drop(columns=["Order Item Profit Ratio", "shipping date (DateOrders)"])

print(f"Columns remaining: {df_clean.shape[1]}")

In [0]:
# Check null counts and percentage across all remaining columns
import pandas as pd
null_summary = pd.DataFrame({
    "null_count": df_clean.isnull().sum(),
    "null_pct": (df_clean.isnull().sum() / len(df_clean) * 100).round(2)
})

# Only show columns that actually have nulls
print(null_summary[null_summary["null_count"] > 0])

### Target Distribution

First thing I want to see is how balanced the target variable is.
This drives the entire imbalance strategy for modeling

Turns out it's roughly 55/45 which is close enough to balanced that I don't need any class weighting
That simplifies the modeling phase significantly.

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean consistent style for all EDA plots
sns.set_theme(style="whitegrid", palette="muted")

# Target distribution
fig, ax = plt.subplots(figsize=(6, 4))

df_clean["Late_delivery_risk"].value_counts().plot(
    kind="bar",
    ax=ax,
    color=["#2563EB", "#F59E0B"],  # blue for on-time, amber for late
    edgecolor="white"
)

ax.set_title("Target Distribution: Late Delivery Risk", fontsize=13, fontweight="bold")
ax.set_xlabel("0 = On Time  |  1 = Late", fontsize=10)
ax.set_ylabel("Order Count", fontsize=10)
ax.set_xticklabels(["On Time", "Late"], rotation=0)

# Annotate bars with counts
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", 
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=10)

plt.tight_layout()

plt.show()

### Late Delivery Rate by Shipping Mode

Shipping mode is one of the strongest candidates for predicting late delivery.
Same Day and First Class should theoretically have lower late rates.

In [0]:
# Calculate late delivery rate by shipping mode
shipping_risk = (
    df_clean.groupby("Shipping Mode")["Late_delivery_risk"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4))

sns.barplot(
    data=shipping_risk,
    x="Shipping Mode",
    y="Late_delivery_risk",
    palette="muted",
    ax=ax
)

ax.set_title("Late Delivery Rate by Shipping Mode", fontsize=13, fontweight="bold")
ax.set_xlabel("Shipping Mode", fontsize=10)
ax.set_ylabel("Late Delivery Rate", fontsize=10)
ax.set_ylim(0, 1)

# Annotate each bar with the exact rate
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2%}",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=10)

# fig.savefig(os.path.join(figures_dir, "02_late_rate_by_shipping_mode.png"), dpi=150, bbox_inches="tight")
plt.tight_layout()
plt.show()

This was the most surprising result in the EDA. First Class has a 95% late delivery rate, the highest of any shipping mode, which is the opposite of what you'd expect from a premium tier.
Standard Class at 38% is actually the most reliable mode in the dataset.
This suggests the late delivery risk isn't just about speed expectations, it likely reflects how NovaSend allocates resources across shipping tiers.
Shipping Mode will almost certainly be one of the top features in the final model.

### Late Delivery Rate by Market

Market tells us where the order is being delivered geographically.
I'd expect regions with longer shipping distances or weaker logistics infrastructure to show higher late rates.

In [0]:
# Calculate late delivery rate by market
market_risk = (
    df_clean.groupby("Market")["Late_delivery_risk"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4))

sns.barplot(
    data=market_risk,
    x="Market",
    y="Late_delivery_risk",
    palette="muted",
    ax=ax
)

ax.set_title("Late Delivery Rate by Market", fontsize=13, fontweight="bold")
ax.set_xlabel("Market", fontsize=10)
ax.set_ylabel("Late Delivery Rate", fontsize=10)
ax.set_ylim(0, 1)

# Annotate each bar with the exact rate
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2%}",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

Every market lands between 54% and 55% late delivery rate, essentially identical across all five regions.
This is worth noting because it's counterintuitive. You'd expect geography to matter for fulfillment risk.

The likely explanation is that late delivery in this dataset is driven by internal operational factors
like shipping mode and order complexity, not by where the order is going.
Market will likely rank near the bottom of feature importance in the final model.

### Late Delivery Rate by Order Status

Order Status captures where an order is in the fulfillment lifecycle at the time of recording.
Some statuses like SUSPECTED_FRAUD or ON_HOLD may have very different late delivery profiles than COMPLETE orders.
This could be strong signal or it could reflect post-intake information we need to be careful about.

In [0]:
# Calculate late delivery rate by order status
status_risk = (
    df_clean.groupby("Order Status")["Late_delivery_risk"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))

sns.barplot(
    data=status_risk,
    x="Order Status",
    y="Late_delivery_risk",
    palette="muted",
    ax=ax
)

ax.set_title("Late Delivery Rate by Order Status", fontsize=13, fontweight="bold")
ax.set_xlabel("Order Status", fontsize=10)
ax.set_ylabel("Late Delivery Rate", fontsize=10)
ax.set_ylim(0, 1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

# Annotate each bar with the exact rate
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2%}",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

Seven of nine order statuses cluster between 55% and 58%, essentially identical to the global late rate.
CANCELED and SUSPECTED_FRAUD show 0% but that's a structural artifact — orders that never ship cannot be late.

This feature will likely add noise rather than signal in the model.
It may still get included in the pipeline but I'd expect it to rank near the bottom of feature importance
alongside Market.

### Numeric Feature Distributions

Now I want to look at the distribution of the numeric features remaining in the dataset.
The goal here is to catch skewness, outliers, and anything that might need transformation before modeling.

In [0]:
# Select numeric columns excluding the target variable
numeric_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != "Late_delivery_risk"]

print(f"Numeric features to plot: {numeric_cols}")

In [0]:
# Plot distributions for all numeric features in a grid
fig, axes = plt.subplots(3, 4, figsize=(16, 10))

# Flatten axes array for easy iteration
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(
        df_clean[col],
        bins=30,
        color="#2563EB",
        edgecolor="white",
        alpha=0.8
    )
    axes[i].set_title(col, fontsize=9, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Count", fontsize=8)

# Hide the two unused subplot slots in the grid
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Numeric Feature Distributions", fontsize=14, fontweight="bold", y=1.01)


plt.tight_layout()
plt.show()

In [0]:
# Investigate unexpected values in Product Status
print(df_clean["Product Status"].value_counts())
print()
print(f"Min: {df_clean['Product Status'].min()}")
print(f"Max: {df_clean['Product Status'].max()}")

In [0]:
# Product Status has no variance, so I'm dropping the column.
df_clean = df_clean.drop(columns=["Product Status"])

print(f"Columns remaining: {df_clean.shape[1]}")

Several price and sales columns are heavily right skewed including Sales, Order Item Total,
Product Price, Order Item Product Price and Sales per customer. I'll use log transformation
before modeling to prevent the model from being dominated by extreme values.

Order Item Discount Rate shows a uniform distribution across discrete tiers which is intentional
based on how discounts are structured in this dataset.

Product Status was dropped entirely. Every record has a value of 0 meaning every product
is marked as in stock. A feature with zero variance adds nothing to a model.

Days for shipment (scheduled) and Order Item Quantity are clean and ready to use as is.

### Correlation Analysis

Now I want to see how the numeric features correlate with the target variable and with each other.
High correlation between features can cause multicollinearity issues in some models.
More importantly I want to see which numeric features have the strongest relationship with Late_delivery_risk.

In [0]:
# Compute correlation matrix across all remaining numeric features including target
corr_matrix = df_clean.select_dtypes(include=["int64", "float64"]).corr()

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,              # show correlation values in each cell
    fmt=".2f",               # round to 2 decimal places
    cmap="coolwarm",         # red for positive, blue for negative correlation
    center=0,                # center the colormap at zero
    square=True,             # keep cells square
    linewidths=0.5,          # add gridlines between cells
    ax=ax
)

ax.set_title("Correlation Matrix: Numeric Features", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

Days for shipment (scheduled) is the only numeric feature with meaningful correlation
to the target at 0.37. Every other numeric feature is essentially uncorrelated with Late_delivery_risk.
This reinforces that the real predictive signal in this dataset is operational, not financial.

Several feature pairs show near perfect multicollinearity. Sales, Order Item Total and
Sales per customer are essentially the same signal. Order Item Product Price and Product Price
are identical. Keeping redundant features adds noise without adding information.

Sales, Order Item Total and Product Price will be dropped before modeling.

In [0]:
# Drop redundant features identified through correlation analysis
df_clean = df_clean.drop(columns=["Sales", "Order Item Total", "Product Price"])

print(f"Columns remaining: {df_clean.shape[1]}")
print(df_clean.columns.tolist())

In [0]:
df_clean.columns.tolist()

In [0]:
import re

# Clean column names for Delta compatibility
df_clean.columns = [
    re.sub(r'[\s,;{}()\n\t=]+', '_', col).strip('_')
    for col in df_clean.columns
]

In [0]:
df_clean.columns

In [0]:
# Write cleaned data to Unity Catalog as a Delta table
df_clean_spark = spark.createDataFrame(df_clean)

df_clean_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("novasend_fulfillment.processed.data_clean")